# Deploy with Model Data Collection

Deploy the latest foundation model to a separate `collector` deployment with a dedicated Azure ML environment containing `azureml-ai-monitoring`. Notebook `06` and its environment remain unchanged.

This notebook adapts the repository's existing implementation:

- `src/deploy/deploy.py` configures `DataCollector` and `DeploymentCollection`.
- `notebooks/deployments/online/custom_scoring_script/model-1/onlinescoring/score.py` records correlated `model_inputs` and `model_outputs` with `Collector`.
- `notebooks/deployments/online/custom_scoring_script/CPU-online-endpoints-custom-container.ipynb` routes collections through inline `Data` destinations.
- `infra/data-collection.tf` provisions the private `datacollection_adls` datastore, private connectivity, and managed-identity RBAC.

All Azure mutations are guarded by `DEPLOY_FOUNDATION_DATA_COLLECTION`. Traffic promotion is separately guarded and neither switch is modified by this notebook.

In [1]:
from pathlib import Path
import ast
import json
import os
import textwrap

import jwt
import yaml
from azure.ai.ml import MLClient
from azure.ai.ml.constants import AssetTypes, ManagedServiceIdentityType
from azure.ai.ml.entities import (
    CodeConfiguration,
    Data,
    DataCollector,
    DeploymentCollection,
    Environment,
    IdentityConfiguration,
    ManagedIdentityConfiguration,
    ManagedOnlineDeployment,
    ManagedOnlineEndpoint,
)
from azure.core.exceptions import HttpResponseError, ResourceNotFoundError
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

notebook_file = globals().get("__vsc_ipynb_file__")
search_start = (
    Path(notebook_file).resolve().parent
    if notebook_file
    else Path.cwd().resolve()
)
for candidate in (search_start, *search_start.parents):
    if (candidate / ".env.example").is_file() and (candidate / "outputs").is_dir():
        WORKSHOP_ROOT = candidate
        break
else:
    raise FileNotFoundError("Run this notebook from inside the workshop folder")
load_dotenv(WORKSHOP_ROOT / ".env", override=True)

credential = AzureCliCredential(tenant_id=os.getenv("AZURE_TENANT_ID") or None)
ml_client = MLClient(
    credential,
    os.environ["AZURE_SUBSCRIPTION_ID"],
    os.environ["AZURE_RESOURCE_GROUP"],
    os.environ["AZUREML_WORKSPACE_NAME"],
)
access_token = credential.get_token("https://management.azure.com/.default")
token_claims = jwt.decode(access_token.token, options={"verify_signature": False})
ORCHESTRATOR_OBJECT_ID = token_claims.get("oid", "<COMPUTE_INSTANCE_UMI_OBJECT_ID>")

ENDPOINT_NAME = os.environ["WORKSHOP_ENDPOINT_NAME"]
MODEL_NAME = os.environ["WORKSHOP_MODEL_NAME"]
DEPLOYMENT_NAME = (
    os.getenv("WORKSHOP_DATA_COLLECTION_DEPLOYMENT_NAME") or "collector"
).strip()
MONITORING_ENVIRONMENT_NAME = (
    os.getenv("WORKSHOP_DATA_COLLECTION_ENVIRONMENT_NAME")
    or "workshop-taxi-data-collection-environment"
).strip()
DATASTORE_NAME = (
    os.getenv("AZUREML_DATA_COLLECTION_DATASTORE") or "datacollection_adls"
).strip()
INSTANCE_TYPE = os.environ["AZUREML_ONLINE_INSTANCE_TYPE"]
PUBLIC_ACCESS = os.getenv(
    "AZUREML_ONLINE_ENDPOINT_PUBLIC_NETWORK_ACCESS", "disabled"
)
IDENTITY_ID = os.getenv("AZUREML_ONLINE_ENDPOINT_IDENTITY_ID", "").strip()
SAMPLING_RATE = float(
    os.getenv("WORKSHOP_DATA_COLLECTION_SAMPLING_RATE", "1.0")
)
ROLLING_RATE = os.getenv(
    "WORKSHOP_DATA_COLLECTION_ROLLING_RATE", "hour"
).strip().lower()
DEPLOY = os.getenv(
    "DEPLOY_FOUNDATION_DATA_COLLECTION", "false"
).lower() in {"1", "true", "yes"}
PROMOTE = os.getenv(
    "PROMOTE_FOUNDATION_DATA_COLLECTION_TRAFFIC", "false"
).lower() in {"1", "true", "yes"}

## Validate the Existing Infrastructure

This read-only check confirms that the target workspace, latest foundation model, and Terraform-provisioned `datacollection_adls` datastore exist. It also validates sampling and rolling settings before any environment or deployment is created.

The endpoint user-assigned identity provisioned by Terraform has `Storage Blob Data Contributor` on the collection account. Keep `AZUREML_ONLINE_ENDPOINT_IDENTITY_ID` configured when creating a new endpoint.

In [ ]:
if not 0.0 < SAMPLING_RATE <= 1.0:
    raise ValueError("WORKSHOP_DATA_COLLECTION_SAMPLING_RATE must be in (0, 1]")
if ROLLING_RATE not in {"minute", "hour", "day"}:
    raise ValueError(
        "WORKSHOP_DATA_COLLECTION_ROLLING_RATE must be minute, hour, or day"
    )

workspace = ml_client.workspaces.get(os.environ["AZUREML_WORKSPACE_NAME"])
datastore = ml_client.datastores.get(DATASTORE_NAME)
registered_model = ml_client.models.get(MODEL_NAME, label="latest")

print(
    {
        "workspace": workspace.name,
        "endpoint": ENDPOINT_NAME,
        "deployment": DEPLOYMENT_NAME,
        "model": f"{registered_model.name}:{registered_model.version}",
        "monitoring_environment": MONITORING_ENVIRONMENT_NAME,
        "data_collection_datastore": datastore.name,
        "data_collection_account": getattr(datastore, "account_name", None),
        "sampling_rate": SAMPLING_RATE,
        "rolling_rate": ROLLING_RATE,
        "deployment_enabled": DEPLOY,
        "traffic_promotion_enabled": PROMOTE,
    }
)

## Create a Separate Monitoring Environment and Scorer

The monitoring package belongs only in this deployment's environment. The generated Conda specification adds `azureml-ai-monitoring` to the normal inference dependencies, while the generated scorer adapts the repository's existing `Collector` pattern.

Only the two approved numeric model features and the prediction are collected. Raw headers, credentials, and unrelated request fields are not persisted. The input collector returns correlation context that links each output record to its input record.

In [ ]:
generated_dir = WORKSHOP_ROOT / "outputs/generated/foundations/model_data_collection"
generated_code_dir = generated_dir / "code"
generated_code_dir.mkdir(parents=True, exist_ok=True)
(generated_code_dir / ".amlignore").write_text(
    "__pycache__/\n*.py[cod]\n", encoding="utf-8"
)
score_path = generated_code_dir / "score.py"
conda_path = generated_dir / "conda.yaml"
request_path = generated_dir / "request.json"

score_source = r'''
import json
import os
from pathlib import Path

import pandas as pd
from azureml.ai.monitoring import Collector

_model = None
_inputs_collector = None
_outputs_collector = None


def init():
    global _model, _inputs_collector, _outputs_collector
    model_root = Path(os.environ["AZUREML_MODEL_DIR"])
    matches = list(model_root.rglob("model.json"))
    if len(matches) != 1:
        raise RuntimeError(f"Expected one model.json, found {len(matches)}")
    _model = json.loads(matches[0].read_text(encoding="utf-8"))
    _inputs_collector = Collector(name="model_inputs")
    _outputs_collector = Collector(name="model_outputs")


def run(raw_data):
    payload = json.loads(raw_data) if isinstance(raw_data, (str, bytes)) else raw_data
    input_data = payload.get("input_data", {})
    columns = input_data.get("columns")
    rows = input_data.get("data")
    if columns != _model["features"]:
        raise ValueError(f"Expected columns in this order: {_model['features']}")
    if not isinstance(rows, list) or not rows:
        raise ValueError("Request must contain at least one row")

    frame = pd.DataFrame(rows, columns=columns).apply(pd.to_numeric, errors="raise")
    collection_context = _inputs_collector.collect(frame.copy())

    predictions = _model["intercept"]
    for feature, coefficient in zip(
        _model["features"], _model["coefficients"], strict=True
    ):
        predictions = predictions + frame[feature] * coefficient

    output_frame = pd.DataFrame(
        {"prediction": predictions.astype(float)}
    )
    _outputs_collector.collect(output_frame, collection_context)
    return {"predictions": output_frame["prediction"].tolist()}
'''
score_source = textwrap.dedent(score_source).lstrip()
ast.parse(score_source, filename=str(score_path))
score_path.write_text(score_source, encoding="utf-8")

conda_source = textwrap.dedent("""
name: azureml-workshop-foundation-data-collection
channels:
  - conda-forge
dependencies:
  - python=3.11
  - pip
  - pip:
      - azureml-inference-server-http==1.4.1
      - azureml-ai-monitoring~=0.1.0b1
      - pandas==2.2.3
""").lstrip()
yaml.safe_load(conda_source)
conda_path.write_text(conda_source, encoding="utf-8")

request = {
    "input_data": {
        "columns": ["tripDistance", "passengerCount"],
        "data": [[2.5, 1], [7.0, 2]],
    }
}
request_path.write_text(json.dumps(request, indent=2) + "\n", encoding="utf-8")

environment_definition = Environment(
    name=MONITORING_ENVIRONMENT_NAME,
    image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04:latest",
    conda_file=str(conda_path),
    description="Separate foundation scoring environment with Azure ML model data collection",
    tags={
        "workshop": "azureml-h2o",
        "purpose": "model-data-collection",
    },
)

print(f"Generated scoring script: {score_path}")
print(f"Generated Conda file: {conda_path}")
print(f"Separate environment: {environment_definition.name}")

## Configure Model Data Collection

The deployment uses the repository's established collection names, `model_inputs` and `model_outputs`. Each collection points to its own immutable path beneath the Terraform-provisioned `datacollection_adls` datastore.

The installed SDK requires the string `"true"` for `DeploymentCollection.enabled`; a Python boolean fails schema serialization in `azure-ai-ml==1.28.1`. The collector samples the configured share of requests and rolls JSONL files hourly.

In [ ]:
def collection_destination(collection_name: str) -> DeploymentCollection:
    path = (
        f"azureml://datastores/{DATASTORE_NAME}/paths/modelDataCollector/"
        f"{ENDPOINT_NAME}/{DEPLOYMENT_NAME}/{collection_name}/"
    )
    return DeploymentCollection(
        enabled="true",
        data=Data(
            name=f"{ENDPOINT_NAME}-{DEPLOYMENT_NAME}-{collection_name}",
            type=AssetTypes.URI_FOLDER,
            path=path,
        ),
    )

collections = {
    "model_inputs": collection_destination("model_inputs"),
    "model_outputs": collection_destination("model_outputs"),
}
data_collector = DataCollector(
    collections=collections,
    sampling_rate=SAMPLING_RATE,
    rolling_rate=ROLLING_RATE,
)

collector_payload = data_collector._to_dict()
for collection_name, collection in collector_payload["collections"].items():
    assert collection["enabled"] == "true"
    assert collection["data"]["path"].startswith(
        f"azureml://datastores/{DATASTORE_NAME}/"
    )
    print(f"{collection_name}: {collection['data']['path']}")
print(f"Sampling rate: {collector_payload['sampling_rate']}")
print(f"Rolling rate: {collector_payload['rolling_rate']}")

## Deploy and Test the Collector Slot

This is the only section that can create or update Azure resources. With deployment enabled, it registers the separate monitoring environment, creates or reuses the foundation endpoint, deploys the `collector` slot, invokes that slot directly, and verifies that both collections remain configured.

Traffic changes only when `PROMOTE_FOUNDATION_DATA_COLLECTION_TRAFFIC=true`. Neither switch is changed by this notebook.

In [ ]:
identity = None
if IDENTITY_ID:
    identity = IdentityConfiguration(
        type=ManagedServiceIdentityType.USER_ASSIGNED,
        user_assigned_identities=[
            ManagedIdentityConfiguration(resource_id=IDENTITY_ID)
        ],
    )

endpoint_definition = ManagedOnlineEndpoint(
    name=ENDPOINT_NAME,
    description="Azure ML workshop foundation endpoint",
    auth_mode="aad_token",
    identity=identity,
    public_network_access=PUBLIC_ACCESS,
    tags={"workshop": "azureml-h2o", "purpose": "model-data-collection"},
)

if DEPLOY:
    registered_environment = ml_client.environments.create_or_update(
        environment_definition
    )
    verified_environment = ml_client.environments.get(
        MONITORING_ENVIRONMENT_NAME,
        version=registered_environment.version,
    )
    assert verified_environment.name == MONITORING_ENVIRONMENT_NAME
    assert verified_environment.version == registered_environment.version
    print(
        f"Registered monitoring environment: "
        f"{verified_environment.name}:{verified_environment.version}"
    )

    deployment_definition = ManagedOnlineDeployment(
        name=DEPLOYMENT_NAME,
        endpoint_name=ENDPOINT_NAME,
        model=registered_model,
        environment=verified_environment,
        code_configuration=CodeConfiguration(
            code=str(generated_code_dir),
            scoring_script="score.py",
        ),
        instance_type=INSTANCE_TYPE,
        instance_count=1,
        app_insights_enabled=True,
        data_collector=data_collector,
    )

    try:
        endpoint = ml_client.online_endpoints.get(ENDPOINT_NAME)
        print(f"Using existing endpoint: {endpoint.name}")
    except ResourceNotFoundError:
        if not IDENTITY_ID:
            raise ValueError(
                "AZUREML_ONLINE_ENDPOINT_IDENTITY_ID is required when creating "
                "an endpoint that writes to the private data-collection datastore"
            )
        try:
            endpoint = ml_client.online_endpoints.begin_create_or_update(
                endpoint_definition
            ).result()
        except HttpResponseError as error:
            if error.status_code == 403:
                raise PermissionError(
                    "The compute-instance UMI cannot assign the endpoint UMI. "
                    "Grant Managed Identity Operator on the endpoint identity to "
                    f"principal {ORCHESTRATOR_OBJECT_ID}."
                ) from error
            raise
        print(f"Created endpoint: {endpoint.name}")

    deployment = ml_client.online_deployments.begin_create_or_update(
        deployment_definition
    ).result()
    if deployment.provisioning_state != "Succeeded":
        raise RuntimeError(f"Deployment state is {deployment.provisioning_state}")

    raw_response = ml_client.online_endpoints.invoke(
        endpoint_name=ENDPOINT_NAME,
        deployment_name=DEPLOYMENT_NAME,
        request_file=str(request_path),
    )
    response = json.loads(raw_response)
    assert response["predictions"] == [9.2, 20.2]
    print(response)

    configured_deployment = ml_client.online_deployments.get(
        DEPLOYMENT_NAME,
        ENDPOINT_NAME,
    )
    assert set(configured_deployment.data_collector.collections) == {
        "model_inputs",
        "model_outputs",
    }
    print(f"Model data collection enabled on: {DEPLOYMENT_NAME}")
    print("Collection is asynchronous; allow several minutes for JSONL files.")

    if PROMOTE:
        endpoint = ml_client.online_endpoints.get(ENDPOINT_NAME)
        endpoint.traffic = {DEPLOYMENT_NAME: 100}
        ml_client.online_endpoints.begin_create_or_update(endpoint).result()
        print(f"Traffic promoted to {DEPLOYMENT_NAME}")
    else:
        print("Traffic remains unchanged.")
else:
    print(f"Prepared collector deployment: {ENDPOINT_NAME}/{DEPLOYMENT_NAME}")
    print(
        "Deployment disabled. Set DEPLOY_FOUNDATION_DATA_COLLECTION=true "
        "in workshop/.env when ready."
    )

## Expected Result

The separate data-collection deployment reaches `Succeeded`, direct invocation returns `[9.2, 20.2]`, and Azure ML asynchronously writes correlated JSONL records beneath the printed `model_inputs` and `model_outputs` datastore paths.

Collection files can take several minutes to appear. The deployment keeps its own monitoring environment and does not modify the environment used by notebook `06`.

Next: `../02_jobs_and_pipelines/01_submit_single_step_merge.ipynb`.